# Ensembles

Two models that are wrong in *different* ways partially cancel. So the useful
question is not "which of my models is best" but "how different are my models
from each other" &mdash; and what can you honestly claim about a combination
*before* you spend a submission finding out?

---
### Setup

In [ ]:
#@title Getting things all setup...
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec lightgbm matplotlib seaborn pyarrow
!git clone https://github.com/agura-alt/ai4chem_openadmet.git
%cd ai4chem_openadmet

In [ ]:
#@title Imports...

import os, sys
SETUP_DIR = os.path.abspath("Setup")
os.path.isdir(SETUP_DIR) or sys.exit(f"No Setup dir at {SETUP_DIR}; cwd is {os.getcwd()}")

if SETUP_DIR not in sys.path:
    sys.path.insert(0, SETUP_DIR)

assert os.path.exists("Setup/common.py") and os.path.getsize("Setup/common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)            # force a fresh read
import common

Change `your-pair-name` to your team name. It has to match the list of registered teams exactly, and be the same in every notebook &mdash; that is what links your work together.

In [ ]:
# Same folder as every other notebook -- splits, predictions, scores.
common.setup(pair="your-pair-name")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style("whitegrid")

test = common.load_test()

# This notebook works from two things you already made: the submission files
# (each one a saved prediction vector) and the scores you logged for them.
display(common.submissions_prepared()[["model", "expected_ma_rae", "split", "why"]])
common.score_matrix().round(3)

Is that first table empty? Then go and run another notebook first &mdash;
`01_validation` saves a LightGBM baseline, `Descriptors` saves whichever
representation won, `Multitask` saves a Chemprop model. You need at least two,
and they are worth more here the more *different* they are.

Preparing those files was free, so nothing has been spent to get to this point.
Which is the question this card turns on: what is the cheapest thing you can do
to a pile of finished models to get one better model?

In [ ]:
# read_submission gives back (predictions, the metadata in its "#" header).
submissions = common.submissions_prepared()
if submissions.empty:
    raise RuntimeError(
        "No submission files in your folder yet. Run 01_validation, Descriptors "
        "or Multitask first -- this notebook combines what they leave behind.")

models = {}
for _, row in submissions.iterrows():        # oldest first, so the newest wins
    preds, meta = common.read_submission(row["path"])
    name = row["model"]
    if name in models:
        print(f"  (replacing an earlier '{name}')")
    models[name] = preds

NAMES = list(models)
print(f"loaded {len(NAMES)}:", ", ".join(NAMES))

---
## 1. How different are your models?

Correlate the members' predictions with each other, one endpoint at a time.

Two models at 0.99 average to almost exactly themselves; two at 0.85 disagree
somewhere. So which pair would you rather average &mdash; and what else would
you want to know before committing to that answer? Does a pair that agrees on
`LogD` still agree on your data-poorest endpoint?

In [ ]:
ENDPOINT = "LogD"        # <-- run this again on your data-poorest endpoint

matrix = pd.DataFrame({n: models[n].set_index("Molecule Name")[ENDPOINT] for n in NAMES})
if matrix.shape[1] < 2:
    print("Only one model loaded, so there is nothing to correlate yet.")
else:
    corr = matrix.corr()
    fig, ax = plt.subplots(figsize=(1.2 * len(NAMES) + 3, 1.0 * len(NAMES) + 2))
    sns.heatmap(corr, annot=True, fmt=".3f", cmap="RdBu_r",
                vmin=min(0.7, float(corr.min().min())), vmax=1, ax=ax)
    ax.set_title(f"Agreement between your models ({ENDPOINT})")
    plt.tight_layout(); plt.show()

### &#9654;&#65039; Predict first

**Do you think your ensemble will be the average of the members' performances, at its best single member, or
better than its best member?**


> `your prediction:`

---
## 2. Three ways to combine

In [ ]:
def weights_for(names, weights=None):
    """Non-negative weights summing to 1 -- the condition section 3 needs."""
    w = np.ones(len(names)) if weights is None else np.asarray(weights, float)
    if len(w) != len(names):
        raise ValueError(f"{len(names)} members but {len(w)} weights")
    if (w < 0).any() or w.sum() <= 0:
        raise ValueError("weights must be non-negative and not all zero")
    return w / w.sum()


def average(names, weights=None):
    """Plain (or weighted) mean of the members' predictions."""
    w = weights_for(names, weights)
    frames = [models[n].set_index("Molecule Name")[common.ENDPOINTS] for n in names]
    shared = frames[0].index
    for frame in frames[1:]:
        shared = shared.intersection(frame.index)   # only molecules everyone predicted
    stacked = sum(frame.loc[shared] * weight for frame, weight in zip(frames, w))
    return stacked.reset_index()


def per_endpoint_best(names, choice):
    """A different member per endpoint. choice maps endpoint -> member name;
    any endpoint you leave out comes from names[0]."""
    out = models[names[0]].set_index("Molecule Name")[common.ENDPOINTS].copy()
    for endpoint, n in choice.items():
        out[endpoint] = models[n].set_index("Molecule Name")[endpoint]
    return out.reset_index()

What other ways could you combine models? Are there certain types of molecules
where one model performs better than another? Are there certain endpoints?

---
## 3. What can you claim about the ensemble's score?

Here is the awkward part. Those files hold predictions for the **test**
molecules only, and you have no labels for those &mdash; so there is nothing in
them you can score directly. `prepare_submission` will not let you submit a
number you never measured. Are you stuck?

No, because absolute error is convex. For weights that are non-negative and sum
to 1:

```
MAE(w1*p1 + w2*p2)  <=  w1*MAE(p1) + w2*MAE(p2)
```

Every endpoint's RAE divides that MAE by the same constant &mdash; the error of
predicting the validation set's own mean &mdash; so the inequality survives all
the way into MA-RAE. **The weighted mean of your members' scores is an upper
bound on the ensemble's score, on that same split.**

In [ ]:
MEMBERS = NAMES[:2]      # <-- choose deliberately, not just the first two
WEIGHTS = None           # equal weights; e.g. [0.6, 0.4] in the same order
MODEL   = "ensemble"     # what this notebook's model is called in your table

if MODEL in MEMBERS:     # an earlier ensemble of yours can be a member, but
    raise ValueError(    # not under the same name, or it overwrites its score
        f"{MODEL!r} is one of its own members. Give this one a new name, "
        "e.g. MODEL = 'ensemble-2'.")

# A member only counts toward the bound if you scored it on the SAME split as
# the others, over all nine endpoints -- a partial score averages a different,
# easier or harder set of endpoints, so it is not comparable.
logged = common.list_scores()
logged = logged[~logged["note"].fillna("").str.contains("partial")]
member_scores = (logged[logged["model"].isin(MEMBERS)]
                 .pivot_table(index="model", columns="split", values="ma_rae")
                 .reindex(MEMBERS))
display(member_scores.round(3))

W = weights_for(MEMBERS, WEIGHTS)
usable = [s for s in member_scores.columns if member_scores[s].notna().all()]
absent = [m for m in MEMBERS if member_scores.loc[m].isna().all()]

print("splits where every member has a full score:", ", ".join(usable) or "(none)")
if absent:
    print("no full-endpoint score logged for:", ", ".join(map(str, absent)),
          "\n  -> go back to the notebook that built it and score it on a split "
          "the others share.")

---
## Score this model against several splits

One number from one split is thin evidence. Which of your splits does this
combination look best on, and is that the split you actually trust? If the
bound moves a lot from one split to the next, what is that telling you about
the members?

**A cross-validation scheme is not a different kind of thing here.** If a
member logged `cv-cluster` as the mean over five folds, then the weighted mean
of those means still bounds the ensemble's mean over the same five folds. The
label is simply whatever you logged the members under.

One thing the bound quietly assumes: that every member's score under a label
came from the *same* validation molecules. What happens to the guarantee if you
logged two models under `random` from two different seeds? Keep at least one
label the same across every notebook today, or the rows in your table are not
comparable in the first place.

** Note: ** every other notebook logs its row with `common.score(truth, preds,
MODEL, SPLIT)`, which measures the predictions against held-out labels and
records what it measured. This notebook cannot: the members' files cover the
*blinded* test molecules, so there is no truth here to compare anything to.

So the cell below uses the lower-level `common.log_score(MODEL, label, value)`,
which writes a number straight into your table without checking it against
anything. It is the one row in `score_matrix()` today that is not a direct
measurement &mdash; it is a bound, and the bound is only as good as the member
scores it was computed from. Which of these would you rather bet a submission
on: a measured score on a split you half-trust, or a proven bound built out of
scores from a split you do?

In [ ]:
for label in usable:
    bound = float(W @ member_scores.loc[MEMBERS, label].to_numpy())
    best_member = float(member_scores[label].min())
    print(f"  {label:14s} bound = {bound:.3f}   best single member = {best_member:.3f}")
    # log_score, not score(): there is no truth to hand it -- see the note above.
    common.log_score(MODEL, label, bound,
                     note=f"UPPER BOUND, not measured: weighted mean of {MEMBERS}")

if not usable:
    print("Nothing to log yet. Pick MEMBERS that share a split, or go and score\n"
          "the ones that are missing -- without a measured number you cannot\n"
          "prepare a submission.")

common.score_matrix().round(3)

---
## Other things to try

- Weight members by their individual score instead of equally. Does the bound
  improve? Keep the weights non-negative and summing to 1 &mdash; what happens
  to the guarantee if you don't?
- Use a *different* member per endpoint &mdash; your fingerprint model may win
  on efflux while your descriptor model wins on LogD, and `per_endpoint_best`
  does this. But picking a winner is not a weighted average: does the bound
  still cover it? And with nine endpoints and a handful of models, what are you
  fitting when you choose those winners on your own validation fold?
- **Measure** instead of bounding. What would the other notebooks have to save
  for you to score a combination directly rather than bound it? What is
  missing from a file that only covers the test molecules?
- Snapshot ensembling and the Caruana greedy selection method both appeared at
  the top of the real leaderboard. Give it a try!

---
## Save your work

Give it a name you will recognise! An ensemble is just another model, so this
notebook can pick up its own output as a member next time round &mdash; under a
different name.

### Make a submission
Go back and try other members and other weights if you like with your best
combination! Then, make a submission.
Note: the number written onto the file is read from your score table, so it is
the bound you logged above &mdash; do you expect the leaderboard to come in
above it or below it?

In [ ]:
common.score_matrix()

** Note: ** The below cell assumes you combined with non-negative weights that
sum to 1, because that is the only case the bound covers. There are valid
reasons to combine some other way &mdash; `per_endpoint_best`, or a stacked
model &mdash; but then you'll have to write your own version of this cell, and
you'll have to justify the number you submit some other way too.

The other thing to notice: the members you are averaging were each refit on
*every* labelled molecule before they were written out, while the scores behind
the bound came from models fit on one split's training side only. Same recipe,
more data &mdash; which way do you expect that to move the real score?

In [ ]:
# Nothing is fitted here -- the members are already trained, so the model you
# submit is just their predictions recombined. The bound above was only for
# estimating the score.

BEST_MEMBERS = MEMBERS       # <-- the combination that won
BEST_WEIGHTS = WEIGHTS       # <-- e.g. [0.6, 0.4], in the same order
MODEL = "ensemble"           # <-- must match what you logged above
SPLIT = "random"             # <-- one of the labels the bound was logged on

test_preds = average(BEST_MEMBERS, BEST_WEIGHTS)
test_preds = test_preds[test_preds["Molecule Name"].isin(test["Molecule Name"])]
test_preds.head()

In [ ]:
common.prepare_submission(test_preds, MODEL, SPLIT,
                          why=f"weighted mean of {BEST_MEMBERS}; the number is a "
                              f"bound, not a measurement")